# Phase 3.5 — A diffusion upscaler from scratch

Pure-PyTorch pixel-space conditional DDPM. **No diffusers, no VAE, no LoRA, no ControlNet.** The point is to see every piece of the diffusion loop next to the markdown that explains it.

**Goal:** train a tiny U-Net that learns to upscale 16×16 → 64×64 (4×) on STL-10 photographs. Output quality is expected to be poor; the deliverable is the *transparency*, not the visuals.

**Acceptance:** ~5k training steps in <4 hours, monotonic loss after early warmup, recognisable samples on ≥3 held-out inputs.

**This notebook deliberately breaks the package-first rule.** Logic stays inline so the math is legible cell-to-cell.

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import ConcatDataset, DataLoader, Dataset
from torchvision import transforms

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT_DIR = REPO_ROOT / "outputs" / "phase3_5"
OUT_DIR.mkdir(parents=True, exist_ok=True)
STL10_ROOT = REPO_ROOT / "data" / "stl10"
STL10_ROOT.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__}  device={device}")
if torch.cuda.is_available():
    print(f"  {torch.cuda.get_device_name(0)}")

# Reproducibility (training will still be approximately, not bitwise, deterministic)
torch.manual_seed(0)
np.random.seed(0)

## 1. The forward (noising) process

DDPM defines a chain of noising steps that gradually destroy a data sample $x_0$ (a real image) into pure Gaussian noise $x_T$:

$$ x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, I) $$

where $\bar{\alpha}_t = \prod_{s \le t} \alpha_s$ and $\alpha_s = 1 - \beta_s$.

The schedule $\{\beta_t\}_{t=1}^{T}$ controls how quickly noise is added. For $t = 0$, $x_t = x_0$ (no noise). For $t = T$, $\bar{\alpha}_t \approx 0$, so $x_t$ is essentially Gaussian noise.

**The big trick:** the marginal $q(x_t \mid x_0)$ at any timestep $t$ is closed-form. We don't have to apply $T$ small noising steps in series — we can jump directly to $x_t$ in one matrix op. That's what makes training tractable.

In [ ]:
T = 1000  # number of diffusion timesteps


def cosine_schedule(timesteps: int, s: float = 0.008) -> torch.Tensor:
    """Improved DDPM cosine schedule (Nichol & Dhariwal 2021).

    Linear schedules destroy signal too quickly at small image sizes; cosine
    keeps low noise levels around longer, which gives the model more useful
    timesteps to learn from.
    """
    steps = torch.arange(timesteps + 1, dtype=torch.float64)
    f = torch.cos(((steps / timesteps) + s) / (1 + s) * math.pi / 2) ** 2
    alpha_bar = f / f[0]
    betas = 1.0 - (alpha_bar[1:] / alpha_bar[:-1])
    return torch.clip(betas, 0.0001, 0.9999).float()


betas = cosine_schedule(T).to(device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)
sqrt_alpha_bars = torch.sqrt(alpha_bars)
sqrt_one_minus_alpha_bars = torch.sqrt(1.0 - alpha_bars)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(alpha_bars.cpu(), label=r"$\bar{\alpha}_t$  (signal preserved)")
ax.plot(1 - alpha_bars.cpu(), label=r"$1 - \bar{\alpha}_t$  (noise variance)")
ax.set_xlabel("timestep t")
ax.set_title("Cosine noise schedule (T=1000)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
def q_sample(x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor) -> torch.Tensor:
    """x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * noise.

    x0 in (B, C, H, W); t in (B,) integer indices; noise same shape as x0.
    """
    sa = sqrt_alpha_bars[t][:, None, None, None]
    so = sqrt_one_minus_alpha_bars[t][:, None, None, None]
    return sa * x0 + so * noise

## 2. Visualizing the forward process

Take a real image, apply `q_sample` at a handful of timesteps, watch it dissolve into noise. This is what the U-Net will learn to *invert*.

In [ ]:
# Use a single test image to demo. Pulled from our project's test set so we
# don't depend on STL-10 being downloaded yet at this point in the notebook.
from PIL import Image as PILImage

demo_path = REPO_ROOT / "data" / "test_images" / "California_Wildflowers.jpg"
demo_img = PILImage.open(demo_path).convert("RGB").resize((64, 64), PILImage.BICUBIC)
demo_arr = np.asarray(demo_img, dtype=np.float32) / 127.5 - 1.0  # to [-1, 1]
demo_t = torch.from_numpy(demo_arr).permute(2, 0, 1).unsqueeze(0).to(device)

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax, ts in zip(axes, [0, 100, 300, 600, 850, 999], strict=True):
    eps = torch.randn_like(demo_t)
    xt = q_sample(demo_t, torch.tensor([ts], device=device), eps)
    img = (xt[0].clamp(-1, 1).cpu().permute(1, 2, 0).numpy() + 1) / 2
    ax.imshow(img)
    ax.set_title(f"t={ts}")
    ax.axis("off")
fig.suptitle("Forward diffusion: one image, six noise levels", fontsize=12)
plt.tight_layout()
plt.show()

## 3. The (LR, HR) dataset

We need a stream of paired (LR, HR) examples. Strategy:
- Take STL-10 photos at 96×96 (an unsupervised photo dataset that ships with torchvision — no HF auth, no manual setup).
- Random-crop to 64×64 → that's HR.
- Bicubic-downsample HR to 16×16 → that's LR.
- Bicubic-upsample LR back to 64×64 → that's the *conditioning image* the U-Net will see, concatenated to the noisy HR as extra channels.

STL-10 has 5k labeled train + 8k labeled test = 13k images. We use both as unlabeled training data (we ignore the class labels). Each image is seen ~6 times across 5k×16 = 80k samples.

We hold out a few STL-10 test images to use for the final qualitative samples — no overlap with the training set.

In [ ]:
# Download STL-10 (skip if already cached; ~150 MB on first run for train+test).
stl_train = torchvision.datasets.STL10(
    root=str(STL10_ROOT), split="train", download=True, transform=None
)
stl_test = torchvision.datasets.STL10(
    root=str(STL10_ROOT), split="test", download=True, transform=None
)
print(f"STL-10 train: {len(stl_train)}   test: {len(stl_test)}")


class PairedDataset(Dataset):
    """Yields (LR_upsampled, HR) tensors in [-1, 1], 3xHxW each.

    Every fetch: random-crop a 64x64 patch, downsample to 16x16, upsample back to 64x64.
    """

    HR_SIZE = 64
    LR_SIZE = 16

    def __init__(self, base):
        self.base = base
        self.crop = transforms.RandomCrop(self.HR_SIZE)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, _label = self.base[idx]
        hr_pil = self.crop(img)
        lr_pil = hr_pil.resize((self.LR_SIZE, self.LR_SIZE), PILImage.BICUBIC)
        lr_up_pil = lr_pil.resize((self.HR_SIZE, self.HR_SIZE), PILImage.BICUBIC)

        def _to_tensor(pil):
            arr = np.asarray(pil, dtype=np.float32) / 127.5 - 1.0
            return torch.from_numpy(arr).permute(2, 0, 1)

        return _to_tensor(lr_up_pil), _to_tensor(hr_pil)


train_ds = PairedDataset(ConcatDataset([stl_train, stl_test]))
train_loader = DataLoader(
    train_ds, batch_size=16, shuffle=True, num_workers=2, drop_last=True, pin_memory=True
)

# Hold-out 3 test images for final qualitative samples (no augmentation).
HOLDOUT_IDX = [42, 137, 911]
holdout_pairs = [train_ds[i] for i in HOLDOUT_IDX]
print(f"train pairs: {len(train_ds)}   batches/epoch: {len(train_loader)}")

## 4. The U-Net

A small pixel-space U-Net (~25M parameters):
- **Input:** 6 channels — 3 noisy HR + 3 LR-bicubic-upsampled. Conditioning by concatenation is the simplest form of conditioning a diffusion model: the LR information is just *there* at every timestep, no cross-attention or ControlNet.
- **Output:** 3 channels — predicted noise $\hat\varepsilon$.
- **Time conditioning:** sinusoidal embedding of $t$, then a 2-layer MLP, broadcast into every residual block.
- **Spatial structure:** 4 stages, downsampling 64 → 32 → 16 → 8, channel mults (1, 2, 4, 4) on a base of 96. Skip connections from each encoder stage to the matching decoder stage.

Real diffusers have attention layers, group-norm tweaks, scale-shift conditioning, and many other refinements. We deliberately leave those out — the bare minimum still works, and the bare minimum is the point of this phase.

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim: int, mlp_dim: int):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim),
            nn.SiLU(),
            nn.Linear(mlp_dim, mlp_dim),
        )

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device, dtype=torch.float32) / half
        )
        args = t.float()[:, None] * freqs[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        return self.mlp(emb)


class ResBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, time_dim: int):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_proj(t_emb)[..., None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class TinyUNet(nn.Module):
    """4-stage pixel-space U-Net.

    Encoder pushes 2 skips per stage (after each ResBlock). 3 downsamples
    between 4 stages. Decoder mirrors: at each stage, concat skips first,
    then upsample to feed the next stage. Last decoder stage skips the
    upsample (we're already back at full resolution).
    """

    def __init__(self, in_ch: int = 6, out_ch: int = 3, base: int = 96, channel_mults=(1, 2, 4, 4)):
        super().__init__()
        time_dim = base * 4
        self.time_emb = TimeEmbedding(base, time_dim)
        self.in_conv = nn.Conv2d(in_ch, base, 3, padding=1)

        # Encoder
        self.down_blocks = nn.ModuleList()
        self.down_samplers = nn.ModuleList()
        skip_chs: list[int] = []  # per-skip channel counts, in push order
        ch = base
        for i, mult in enumerate(channel_mults):
            out = base * mult
            self.down_blocks.append(
                nn.ModuleList([ResBlock(ch, out, time_dim), ResBlock(out, out, time_dim)])
            )
            skip_chs += [out, out]
            ch = out
            # Downsample after every stage except the last (between-stage transitions).
            if i < len(channel_mults) - 1:
                self.down_samplers.append(nn.Conv2d(ch, ch, 4, stride=2, padding=1))

        # Bottleneck
        self.mid1 = ResBlock(ch, ch, time_dim)
        self.mid2 = ResBlock(ch, ch, time_dim)

        # Decoder — mirrors encoder. up_samplers transitions BETWEEN stages.
        self.up_blocks = nn.ModuleList()
        self.up_samplers = nn.ModuleList()
        for i, mult in enumerate(reversed(channel_mults)):
            out = base * mult
            skip2 = skip_chs.pop()  # most recent skip (consumed first in decoder)
            skip1 = skip_chs.pop()
            self.up_blocks.append(
                nn.ModuleList(
                    [ResBlock(ch + skip2, out, time_dim), ResBlock(out + skip1, out, time_dim)]
                )
            )
            ch = out
            if i < len(channel_mults) - 1:
                self.up_samplers.append(nn.ConvTranspose2d(ch, ch, 4, stride=2, padding=1))

        self.out_norm = nn.GroupNorm(8, base)
        self.out_conv = nn.Conv2d(base, out_ch, 3, padding=1)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_emb = self.time_emb(t)
        h = self.in_conv(x)

        skips: list[torch.Tensor] = []
        n_stages = len(self.down_blocks)
        for i, (b1, b2) in enumerate(self.down_blocks):
            h = b1(h, t_emb)
            skips.append(h)
            h = b2(h, t_emb)
            skips.append(h)
            if i < n_stages - 1:
                h = self.down_samplers[i](h)

        h = self.mid1(h, t_emb)
        h = self.mid2(h, t_emb)

        for i, (b1, b2) in enumerate(self.up_blocks):
            # At entry, h is at this stage's spatial size (matches the skips here).
            h = b1(torch.cat([h, skips.pop()], dim=1), t_emb)
            h = b2(torch.cat([h, skips.pop()], dim=1), t_emb)
            if i < n_stages - 1:
                h = self.up_samplers[i](h)

        return self.out_conv(F.silu(self.out_norm(h)))


model = TinyUNet().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"U-Net parameters: {n_params / 1e6:.1f}M")

# Quick shape sanity check before the long training run.
with torch.no_grad():
    _x = torch.randn(2, 6, 64, 64, device=device)
    _t = torch.randint(0, T, (2,), device=device)
    _y = model(_x, _t)
    assert _y.shape == (2, 3, 64, 64), _y.shape
print(f"forward shape OK: {_y.shape}")

## 5. The training objective — predict the noise

The classical DDPM training step is comically simple:

1. Sample a real image $x_0$ and a timestep $t \sim \mathrm{Uniform}\{1,\dots,T\}$.
2. Sample noise $\varepsilon \sim \mathcal{N}(0, I)$.
3. Make $x_t$ via the closed-form forward step (one matmul).
4. Run the U-Net with input $[x_t \, ; \, \text{LR\_upsampled}]$ and timestep $t$, getting $\hat\varepsilon$.
5. Loss: $\|\hat\varepsilon - \varepsilon\|_2^2$.

That's it. The model isn't predicting the clean image, it isn't predicting the next $x_{t-1}$ — it's predicting the noise that was added. From that prediction, sampling later can recover *any* of those equivalent quantities via simple algebra. This noise-prediction parameterisation (called $\varepsilon$-pred) is what made DDPM stable enough to train at scale.

In [ ]:
TRAIN_STEPS = 5000
LOG_EVERY = 50

optim = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.0)
scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None

loss_history: list[tuple[int, float]] = []

model.train()
step = 0
data_iter = iter(train_loader)
while step < TRAIN_STEPS:
    try:
        lr_up, hr = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        lr_up, hr = next(data_iter)

    lr_up = lr_up.to(device, non_blocking=True)
    hr = hr.to(device, non_blocking=True)
    bsz = hr.shape[0]
    t = torch.randint(0, T, (bsz,), device=device)
    eps = torch.randn_like(hr)
    x_t = q_sample(hr, t, eps)
    inp = torch.cat([x_t, lr_up], dim=1)

    optim.zero_grad(set_to_none=True)
    if scaler is not None:
        with torch.amp.autocast("cuda", dtype=torch.float16):
            eps_pred = model(inp, t)
            loss = F.mse_loss(eps_pred, eps)
        scaler.scale(loss).backward()
        scaler.step(optim)
        scaler.update()
    else:
        eps_pred = model(inp, t)
        loss = F.mse_loss(eps_pred, eps)
        loss.backward()
        optim.step()

    if step % LOG_EVERY == 0:
        loss_history.append((step, float(loss.item())))
        if step % 500 == 0:
            print(f"step {step:>5}  loss={loss.item():.4f}")
    step += 1

print(f"\ntraining done. final loss: {loss_history[-1][1]:.4f}")

## 6. Loss curve

We expect a noisy decrease — diffusion losses are inherently variable because of the random timestep sampling — but the trend should clearly drop from the early-warmup plateau. A flat or rising curve means something is broken (almost always: schedule alignment, dtype mismatch, or wrong concat ordering).

In [ ]:
steps = np.array([s for s, _ in loss_history])
losses = np.array([loss for _, loss in loss_history])
# 10-point rolling mean to make the trend legible
window = 10
if len(losses) >= window:
    smooth = np.convolve(losses, np.ones(window) / window, mode="valid")
else:
    smooth = losses

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(steps, losses, alpha=0.4, label="raw")
ax.plot(steps[window - 1 :] if len(losses) >= window else steps, smooth, label=f"{window}-pt MA")
ax.set_xlabel("step")
ax.set_ylabel("MSE loss")
ax.set_title("Training loss")
ax.set_yscale("log")
ax.legend()
ax.grid(alpha=0.3)
fig.savefig(OUT_DIR / "loss_curve.png", dpi=110, bbox_inches="tight")
plt.show()

## 7. The reverse process — DDIM sampling

DDPM's training objective is forward — predict the noise that was added. To **generate** an image we run the chain backwards: start with pure Gaussian noise $x_T$ and iteratively denoise to $x_0$.

[DDIM (Song et al. 2020)](https://arxiv.org/abs/2010.02502) gives a deterministic short-circuit: instead of $T=1000$ steps we can pick a sub-sequence of, say, 50 timesteps and walk through them. At each step we use the noise prediction $\hat\varepsilon$ to estimate $\hat x_0$ in one shot, then *re-noise* to the next timestep $\hat x_{t-1}$. With $\eta = 0$ (deterministic), this is fast and identity-preserving — perfect for upscaling, where we don't need stochasticity.

$$ \hat{x}_0 = \frac{x_t - \sqrt{1-\bar\alpha_t}\,\hat\varepsilon}{\sqrt{\bar\alpha_t}}, \qquad x_{t-1} = \sqrt{\bar\alpha_{t-1}}\,\hat x_0 + \sqrt{1-\bar\alpha_{t-1}}\,\hat\varepsilon $$

In [ ]:
@torch.no_grad()
def ddim_sample(
    model: nn.Module,
    lr_up: torch.Tensor,
    num_steps: int = 50,
    seed: int | None = 0,
) -> torch.Tensor:
    """Deterministic DDIM sampler. lr_up is (B, 3, 64, 64), in [-1, 1]."""
    model.eval()
    bsz = lr_up.shape[0]
    timesteps = torch.linspace(T - 1, 0, num_steps + 1, dtype=torch.long, device=device)
    if seed is not None:
        gen = torch.Generator(device=device).manual_seed(seed)
        x = torch.randn(lr_up.shape, generator=gen, device=device)
    else:
        x = torch.randn_like(lr_up)

    for i in range(num_steps):
        t = timesteps[i]
        t_next = timesteps[i + 1]
        ab_t = alpha_bars[t]
        ab_next = alpha_bars[t_next] if t_next > 0 else torch.tensor(1.0, device=device)

        inp = torch.cat([x, lr_up], dim=1)
        eps_hat = model(inp, t.expand(bsz))
        x0_hat = (x - torch.sqrt(1 - ab_t) * eps_hat) / torch.sqrt(ab_t)
        x0_hat = x0_hat.clamp(-1, 1)
        x = torch.sqrt(ab_next) * x0_hat + torch.sqrt(1 - ab_next) * eps_hat

    return x.clamp(-1, 1)

## 8. Sample outputs — recognisable but bad

Three held-out STL-10 images. For each: the LR input (16×16, nearest-neighbour upsampled for display), the model's reconstruction, and the HR ground truth.

The samples won't be impressive in absolute terms — that's the point of *tiny* diffusion. They should, however, be **clearly attempting** the right scene: similar dominant colours, similar coarse structure. That's what the production pipeline (Phases 1–4) is, just with a thousand more lines of engineering.

In [ ]:
# Build the LR + HR tensors for the 3 holdout images
lr_batch = torch.stack([p[0] for p in holdout_pairs]).to(device)
hr_batch = torch.stack([p[1] for p in holdout_pairs]).to(device)

samples = ddim_sample(model, lr_batch, num_steps=50, seed=0)


def to_display(t: torch.Tensor) -> np.ndarray:
    return ((t.detach().cpu().permute(1, 2, 0).numpy() + 1) / 2).clip(0, 1)


fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for r in range(3):
    axes[r, 0].imshow(to_display(lr_batch[r]))
    axes[r, 0].set_title("LR (16×16, nn-upscaled)" if r == 0 else "")
    axes[r, 1].imshow(to_display(samples[r]))
    axes[r, 1].set_title("tiny-diffusion sample" if r == 0 else "")
    axes[r, 2].imshow(to_display(hr_batch[r]))
    axes[r, 2].set_title("HR ground truth" if r == 0 else "")
    # Difference image (absolute error vs HR), useful for diagnostics
    diff = (samples[r] - hr_batch[r]).abs().mean(dim=0).cpu().numpy()
    axes[r, 3].imshow(diff, cmap="hot", vmin=0, vmax=1)
    axes[r, 3].set_title("|sample − HR|" if r == 0 else "")
    for c in range(4):
        axes[r, c].axis("off")

fig.suptitle("Tiny-diffusion samples on 3 held-out STL-10 images", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(OUT_DIR / "samples.png", dpi=120, bbox_inches="tight")
plt.show()

## 9. What this notebook deliberately *doesn't* do

Phases 1–6 of the production pipeline add, in order, all the things this notebook left out. Reading them now should feel transparent — every component below maps to a specific limitation we just observed.

- **Latent space (VAE).** Real diffusers compress images 8× before applying the diffusion model. That's how Stable Diffusion can fit 1024² inputs in 8 GB of VRAM. This notebook stays in pixel space at 64² because the math is more legible there.
- **Captions / text conditioning.** We conditioned by concatenation only — fine for a tile-style model, useless for prompt control. Phase 4 adds BLIP-2-generated captions and CLIP text encoders.
- **ControlNet.** Concat-conditioning is weaker than ControlNet's auxiliary-network conditioning, especially for structural features. Phase 2 swaps to ControlNet Tile.
- **LoRA fine-tuning.** Phase 4 trains a LoRA on a *frozen* SD 1.5 backbone — we get the SD ecosystem's pretraining for free and only fine-tune what's domain-specific.
- **Tiling for full-resolution outputs.** This U-Net is fixed at 64². Phase 2's `tiling.py` lets us run the full pipeline at 1000² by stitching overlapping 512² tiles with feathered blends.
- **Better noise schedules / parameterisations.** `v`-parameterisation, snr-weighted losses, EDM schedules — all small wins that compound.
- **Step-distillation.** Real production inference uses 4–25 steps via consistency models or LCM-distilled schedulers; we used 50 here for clarity.

Total lines of "real" diffusion code in this notebook: ~150. Total lines of code in `src/upscaler/`: 4× that, plus tests. Most of the engineering work in this project is the *plumbing around* the diffusion process, not the diffusion itself.